# GeoSPARQL Extension Functions (rdflib-geosparql)

The following functions are implemented in rdflib-geosparql, but are not (yet) officially standardized by OGC.

## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet pyvista trame --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import json
import pyvista as pv
import trimesh
import shapely.geometry
from shapely import wkt
from shapely.ops import unary_union
from io import BytesIO
from shapely.ops import transform
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, GeoJSON, FullScreenControl, LayersControl, Popup#, Tooltip

mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in resultlist:
            for item in row:
                if isinstance(row[item],Literal) and row[item].datatype!=None:
                    res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip()+"</a></td></tr>"
                elif isinstance(row[item],URIRef):
                    res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
                else:
                    res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip()+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    bboxes=[]
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in rows:
            if resultlist!=None and row in resultlist[0]:
                popup="<h3>"+str(row)+"</h3><ul>"        
                for rowres in resultlist:
                    for item in rowres:
                        if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip()+"</a></li>"
                        elif isinstance(rowres[item],URIRef):
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                        else:
                            popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip()+"</li>"
                popup+="</ul>"
                toprocess=resultlist[0][row].strip()
                if toprocess.startswith("<http"):
                    toprocess=toprocess[toprocess.find(">")+1:]
                geom = wkt.loads(toprocess)
                if not shapely.is_empty(geom):
                    lastcentroid=geom.centroid
                    nlayer=WKTLayer(name=str(row),wkt_string=geom.wkt,hover_style={"fillColor": "red"})
                    bounds=geom.bounds
                    bboxes.append(shapely.geometry.box(bounds[0],bounds[1],bounds[2],bounds[3]))
                    msg=W.HTML()
                    msg.value=popup
                    nlpopup=Popup(
                        location=[lastcentroid.y,lastcentroid.x],
                        child=msg,
                        close_button=True,
                        auto_close=False,
                        close_on_escape_key=False
                    )
                    nlayer.popup=msg
                    layers.append(nlayer)
    if lmap==None:
        maxbounds=unary_union(bboxes).bounds
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
        control = LayersControl(position='topright')
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    lmap.fit_bounds(((maxbounds[1],maxbounds[2]),(maxbounds[3],maxbounds[0])))
    return lmap    

def processQueryResults(qres,rows=[],lmap=None,rows3d=[]):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    if rows3d!=[]:
        if qres is not None and len(qres.bindings)>0:
            resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
            for rw in rows3d:
                print(resultlist)
                if rw in resultlist:
                    tm = trimesh.load(BytesIO(resultlist[rw]), file_type="ply")
                    # Convert Trimesh -> PyVista
                    mesh = pv.wrap(tm)        
                    mesh.plot(notebook=True)
                    #mesh = pv.read("model.ply")
                    #plotter = pv.Plotter(notebook=True)
                    #plotter.add_mesh(mesh)
                    #plotter.show()
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-34if_wjb.log


## GeoSPARQL Serializations

### geofext:asGeoCode Function

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gc
WHERE {
  my:A geo:hasGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeocode(?aLiteral,"http://opengis.net/ont/geocode/OpenLocationCode") as ?gc)
}
"""),["aLiteral"],None)

POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
http://opengis.net/ont/geocode/OpenLocationCode


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gc,2G8PJ822+22


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asGLTF Function

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gltf
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGLTF(?aLiteral) as ?gltf)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gltf,


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asOBJ Function

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?obj
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asOBJ(?aLiteral) as ?obj)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asPLY Function

In [5]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?ply
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asPLY(?aLiteral) as ?ply)
}
"""),["aLiteral"],None,["ply"])

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
ply,


[{'aLiteral': rdflib.term.Literal('\n\n            <http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))\n\n    ', datatype=rdflib.term.URIRef('http://www.opengis.net/ont/geosparql#wktLiteral')), 'ply': rdflib.term.Literal('', datatype=rdflib.term.URIRef('http://www.opengis.net/ont/geosparql#plyLiteral'))}]


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asJSONFG Function

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?jsonfg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asJSONFG(?aLiteral) as ?jsonfg)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asSVG Function

In [7]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?svg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asSVG(?aLiteral) as ?svg)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
svg,


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Directional relation functions

This section introduces directional relation functions in rdflib-geosparql.

### geofext:above Function

Checks if the first input geometry is above the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?abovee) as ?D_above_A) (xsd:boolean(?nabovee) as ?A_above_D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:above(?aLiteral, ?dLiteral) as ?nabovee)
  BIND (geof:above(?dLiteral, ?aLiteral) as ?abovee)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
D_above_A,true
A_above_D,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:behind Function

Checks if the first input geometry is behind the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [9]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_Behind_A) (xsd:boolean(?nbeloww) as ?A_Behind_D)
WHERE {
  BIND("Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?aLiteral)
  BIND("Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:behind(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:behind(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"
dLiteral,"Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"
D_Behind_A,false
A_Behind_D,true


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:inFrontOf Function

Checks if the first input geometry is in front of the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [10]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_InFrontOf_A) (xsd:boolean(?nbeloww) as ?A_InFrontOf_D)
WHERE {
  BIND("Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?aLiteral)
  BIND("Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:inFrontOf(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:inFrontOf(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"
dLiteral,"Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"
D_InFrontOf_A,true
A_InFrontOf_D,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:below Function

Checks if the first input geometry is below the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_Below_A) (xsd:boolean(?nbeloww) as ?A_Below_D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:below(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:below(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
D_Below_A,true
A_Below_D,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:leftOf Function

Checks if the first input geometry is left of the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?leftt) as ?A_leftOf_D) (xsd:boolean(?nleftt) as ?D_LeftOf_A)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:leftOf(?aLiteral, ?dLiteral) as ?leftt)
  BIND (geof:leftOf(?dLiteral, ?aLiteral) as ?nleftt)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
A_leftOf_D,true
D_LeftOf_A,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

### geofext:rightOf Function

Checks if the first input geometry is right of the second input geometry. Geometry coordinate reference systems are normalized if they differ.

In [13]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?rightt) as ?A_rightOf_D) (xsd:boolean(?nrightt) as ?D_rightOf_A)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:rightOf(?aLiteral, ?dLiteral) as ?nrightt)
  BIND (geof:rightOf(?dLiteral, ?aLiteral) as ?rightt)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
A_rightOf_D,true
D_rightOf_A,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

## Geometry Accessor Functions

### geofext:boundingDiagonal Function

Returns the diagonal of a geometry's bounding box, i.e. a 2 point LineString from its minimum to maximum coordinates.

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bdiag
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundingDiagonal(?aLiteral) as ?bdiag)
}
"""),["aLiteral","bdiag"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bdiag,"LINESTRING (-83.6 34.1, -83.2 34.5)"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:endPoint Function ⬜️🧊

Returns the last point of a given geometry in the CRS and literal format of the input geometry.

In [15]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?endPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:endPoint(?literal) AS ?endPoint)
}
"""),["literal","endPoint"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
endPoint,POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:exteriorRing Function

In [16]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?exRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:exteriorRing(?literal) AS ?exRing)
}
"""),["literal","exRing"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
exRing,"LINEARRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:isCCW Function

In [17]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isACW ?isECW {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isCCW(?a_wkt) AS ?isACW)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isCCW(?e_wkt) AS ?isECW)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
e_wkt,"LineString(-83.4 34.0, -83.3 34.3)"


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isCW Function

In [18]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isACW ?isECW {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isCW(?a_wkt) AS ?isACW)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isCW(?e_wkt) AS ?isECW)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
e_wkt,"LineString(-83.4 34.0, -83.3 34.3)"


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isClosed Function

Tests if a geometry's start and endpoint are equal

In [19]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isAClosed ?isEClosed {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isClosed(?a_wkt) AS ?isAClosed)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isClosed(?e_wkt) AS ?isEClosed)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isAClosed,true
e_wkt,"LineString(-83.4 34.0, -83.3 34.3)"
isEClosed,false


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isCollection Function

Tests if a geometry is of a geometry collection type including a MUTLI type

In [20]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?C1 ?C2 ?C3 ?isC1 ?isC2 ?isC3 {
    BIND("GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"^^geo:wktLiteral as ?C1)
    BIND("MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"^^geo:wktLiteral as ?C2)
    BIND("POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"^^geo:wktLiteral as ?C3)
    BIND(geof:isCollection(?C1) AS ?isC1)
    BIND(geof:isCollection(?C2) AS ?isC2)
    BIND(geof:isCollection(?C3) AS ?isC3)
}
"""),["C1","C2","C3"],None)

Variable,Value
C1,"GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"
C2,"MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"
C3,"POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"
isC1,true
isC2,true
isC3,false


Map(center=[2.033333333333333, 2.033333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### geofext:isRectangle Function

In [21]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?isRectangle ?isNoRectangle {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRectangle(?a_wkt) AS ?isNoRectangle)
    BIND(geof:isRectangle(geofo:envelope(?a_wkt)) AS ?isRectangle)
}
"""),["a_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isNoRectangle,true
isRectangle,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:isRing Function

Checks if a geometry is simple and closed

In [22]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?a_wkt ?o_wkt ?isARing ?isORing {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRing(?a_wkt) AS ?isARing)
            <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isRing("POINT M (1 2 3)"^^geo:wktLiteral) AS ?isORing)
}
"""),["a_wkt","o_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isARing,true
o_wkt,"Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
isORing,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:isTriangle Function

In [23]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?isTriangle ?isNoTriangle {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isTriangle(?a_wkt) AS ?isNoTriangle)
    BIND(geof:isTriangle("POLYGON ((0 0,0 1,1 1,0 0))"^^geo:wktLiteral) AS ?isTriangle)
}
"""),["a_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isNoTriangle,false
isTriangle,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:isValid Function

In [24]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?a_isValid ?o_isNotValid {
<http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
BIND(geof:isValid(?a_wkt) AS ?a_isValid)
        <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
BIND(geof:isValid("POLYGON((0 0, 10 10, 0 10, 10 0, 0 0))"^^geo:wktLiteral) AS ?o_isNotValid)
}
"""),["a_wkt","o_wkt"],None)

Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
a_isValid,true
o_wkt,"Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
o_isNotValid,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:isValidTrajectory Function

In [25]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?validTGeom ?isValidT ?O_isValidT2 ?A_isValidT {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isValidTrajectory(?a_wkt) AS ?A_isValidT)
    <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isValidTrajectory(?o_wkt) AS ?O_isValidT2)
    BIND("LineString M(0 0 1, 10 10 2, 0 10 3, 10 0 4, 0 0 5)"^^geo:wktLiteral AS ?validTGeom)
    BIND(geof:isValidTrajectory(?validTGeom) AS ?isValidT)
}
"""),["a_wkt","o_wkt","validTGeom"],None)

[[0.0, 0.0, 1.0], [10.0, 10.0, 2.0], [0.0, 10.0, 3.0], [10.0, 0.0, 4.0], [0.0, 0.0, 5.0]]
curM: -inf
curM: 1.0
curM: 2.0
curM: 3.0
curM: 4.0


Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
A_isValidT,false
o_wkt,"Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
O_isValidT2,false
validTGeom,"LineString M(0 0 1, 10 10 2, 0 10 3, 10 0 4, 0 0 5)"
isValidT,true


Map(center=[5.0, 5.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

### geofext:M Function

Returns the measurement coordinate of a point geometry.

In [26]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?m
WHERE {
  my:PPointGeom geo:asWKT ?literal .
  BIND(geof:M(?literal) AS ?m)
}
"""),["literal"],None)

Variable,Value
literal,Point M (-83.4 34.0 5.0)
m,5.0


Map(center=[34.0, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:maxM Function

Returns the maximum measurement coordinate of a given geometry.

In [27]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:maxM(?literal) AS ?maxM)
}
"""),["literal"],None)

Variable,Value
literal,"LineString M (-83.4 34.0 5, -83.3 34.3 10)"
maxM,10.0


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:minM Function

Returns the minimum measurement coordinate of a geometry.

In [28]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?minM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:minM(?literal) AS ?minM)
}
"""),["literal"],None)

Variable,Value
literal,"LineString M (-83.4 34.0 5, -83.3 34.3 10)"
minM,5.0


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:numInteriorRing Function

In [29]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numInteriorRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numInteriorRing(?literal) AS ?numInteriorRing)
}
"""),["literal"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numInteriorRing,0


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:numPatches Function

In [64]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numPatches
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numPatches(?literal) AS ?numPatches)
}
"""),["literal"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numPatches,1


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:numPoints Function

Returns the number of points in a given geometry.

In [30]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numPoints
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numPoints(?literal) AS ?numPoints)
}
"""),["literal"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numPoints,5


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:pointN Function

Returns the nth point of a given geometry.

In [31]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointN
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:pointN(?literal,"1"^^xsd:integer) AS ?pointN)
}
"""),["literal"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
pointN,POINT (-83.2 34.1)


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:startPoint Function

Returns the first point of a given geometry in the CRS and literal format of the input geometry.

In [32]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?startPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:startPoint(?literal) AS ?startPoint)
}
"""),["literal","startPoint"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
startPoint,POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:X Function

Returns the X coordinate of a point geometry. The X axis is determined from its CRS definition.

In [33]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?x
WHERE {
  my:FExactGeom geo:asWKT ?literal .
  BIND(geof:X(?literal) AS ?x)
}
"""),["literal"],None)

Variable,Value
literal,Point(-83.4 34.4)
x,-83.4


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:Y Function

Returns the Y coordinate of a Point geometry. The Y axis is determined by its CRS definition.

In [34]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?y
WHERE {
  my:FExactGeom geo:asWKT ?literal .
  BIND(geof:Y(?literal) AS ?y)
}
"""),["literal"],None)

Variable,Value
literal,Point(-83.4 34.4)
y,34.4


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:Z Function

In [35]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?z
WHERE {
  my:NPointGeom geo:asWKT ?literal .
  BIND(geof:Z(?literal) AS ?z)
}
"""),["literal"],None)

Variable,Value
literal,Point Z (-77.059 38.9 1)
z,1.0


Map(center=[38.9, -77.059], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## Geometry Measurement Functions

### geofext:azimuth Function

In [36]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?azimuth
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:azimuth(?aLiteral) as ?azimuth)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
azimuth,90.0


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:closestPoint Function

Returns the closest point between two input geometries. This point is the first point of the shortest line between the geometries.

In [37]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?cPoint
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:closestPoint(?aLiteral, ?dLiteral) as ?cPoint)
}
"""),["aLiteral","dLiteral","cPoint"],None)

Variable,Value


Map(center=[-83.2, 34.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


### geofext:farthestCoordinate Function

In [38]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?farthestCoord
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:farthestCoordinate(?aLiteral, ?dLiteral) as ?farthestCoord)
}
"""),["aLiteral","dLiteral","farthestCoord"],None)

Variable,Value


Map(center=[-83.2, 34.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:frechetDistance Function

In [39]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?fdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:frechetDistance(?cLiteral, ?fLiteral) as ?fdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
fdistance,0.41231056256177195


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:hausdorffDistance Function

In [40]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?hdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:hausdorffDistance(?cLiteral, ?fLiteral) as ?hdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
hdistance,0.41231056256177195


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:longestLine Function 🧊

Returns the longest line between two geometries.

In [41]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?longestLine
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:longestLine(?aLiteral, ?dLiteral) as ?longestLine)
}
"""),["aLiteral","dLiteral","longestLine"],None)

LONGESTLINE
0.31622776601683567
0.5099019513592787
0.5099019513592787
0.31622776601683567
0.31622776601683567
0.1414213562373065
0.14142135623731653
0.14142135623731653
0.1414213562373065
0.1414213562373065
0.5099019513592773
0.5099019513592802
0.31622776601683794
0.31622776601683344
0.5099019513592773
0.5830951894845285
0.7071067811865476
0.5830951894845285
0.42426406871192446
0.5830951894845285
0.31622776601683567
0.5099019513592787
0.5099019513592787
0.31622776601683567
0.31622776601683567


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
longestLine,"LINESTRING (-83.6 34.5, -83.1 34)"


Map(center=[34.25, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:maxDistance Function 🧊

Calculates the largest distance between two geometries in the unit of the CRS of the first geometry.

In [42]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?maxDistance
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:maxDistance(?aLiteral, ?dLiteral) as ?maxDistance)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
maxDistance,0.7071067811865476


Map(center=[34.099999999999994, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:minimumClearance Function

In [43]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mclearance
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumClearance(?aLiteral) as ?mclearance)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
mclearance,0.3999999999999915


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:minimumClearanceLine Function

In [44]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mclearancel
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumClearanceLine(?aLiteral) as ?mclearancel)
}
"""),["aLiteral","mclearancel"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
mclearancel,"LINESTRING (-83.6 34.1, -83.2 34.1)"


Map(center=[34.1, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:shortestLine Function 🧊

Calculates the shortest line between two given geometries in the CRS of the first input geometry.

In [45]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?literal2 ?sline
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  my:OExactGeom geo:asWKT ?literal2 .
  BIND(geof:shortestLine(?literal,?literal2) AS ?sline)
}
"""),["literal","literal2","sline"],None)

[(<POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))>, 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'), (<POLYGON ((2.472 47.524, 2.407 47.497, 2.446 47.437, 2.571 47.486, 2.393 47....>, 'http://www.opengis.net/def/crs/OGC/1.3/CRS84')]
POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
POLYGON ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))
LINESTRING (-83.2 34.5, 2.39251 47.44793)


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
literal2,"Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
sline,"LINESTRING (-83.2 34.5, 2.39251 47.44793)"


Map(center=[40.973965, -40.403745], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Geometry Modification Functions

### geofext:force2D Function

Creates a 2D geometry from a given geometry of a higher dimension by reducing its dimensionality

In [46]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force2D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force2D(?literal) AS ?force2D)
}
"""),["literal","force2D"],None)

Variable,Value
literal,"Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
force2D,"POLYGON ((-77.089005 38.913574, -77.029953 38.913574, -77.029953 38.886321, -77.089005 38.886321, -77.089005 38.913574))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### geofext:force3D Function

Creates a 3D geometry by adding a given Z coordinate to every point of the input geometry

In [47]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force3D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force3D(?literal,"5.0"^^xsd:double) AS ?force3D)
}
"""),["literal","force3D"],None)

Variable,Value
literal,"Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### geofext:removeRepeatedPoints Function

In [48]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal  ?rep
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:removeRepeatedPoints(?literal) AS ?rep)
}
"""),["literal"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
rep,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:reverse Function

In [49]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?reverse
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:reverse(?literal) AS ?reverse)
}
"""),["literal","reverse"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
reverse,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:simplify Function

In [50]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?simple
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:simplify(?literal,"1.5"^^xsd:double) AS ?simple)
}
"""),["literal","simple"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
simple,"POLYGON ((-83.6 34.5, -83.2 34.1, -83.2 34.5, -83.6 34.5))"


Map(center=[34.36666666666667, -83.33333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:smooth Function

Calculates a smoothed version of the given geometry using the Chaikin Smooting algorithm

In [51]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?smoothed
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:smooth(?literal,0.1) AS ?smoothed)
}
"""),["literal","smoothed"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
smoothed,"POLYGON ((-83.40625 34.1, -83.39375 34.1, -83.381640625 34.100390625, -83.36992187500002 34.101171875000006, -83.35859375 34.10234375, -83.34765625 34.10390625, -83.337109375 34.105859375, -83.32695312499999 34.108203125, -83.31718750000002 34.110937500000006, -83.3078125 34.1140625, -83.298828125 34.117578125, -83.290234375 34.121484375, -83.28203125 34.12578125, -83.27421875 34.130468750000006, -83.266796875 34.135546875, -83.259765625 34.141015625, -83.25312500000001 34.146875, -83.246875 34.153125, -83.24101562500002 34.159765625000006, -83.235546875 34.166796875, -83.23046875 34.17421875, -83.22578125 34.182031249999994, -83.221484375 34.190234375, -83.217578125 34.198828125000006, -83.2140625 34.2078125, -83.2109375 34.2171875, -83.208203125 34.226953124999994, -83.205859375 34.237109375, -83.20390625 34.247656250000006, -83.20234375000001 34.25859375, -83.201171875 34.269921875, -83.200390625 34.281640624999994, -83.2 34.29375, -83.2 34.306250000000006, -83.20039062500001 34.318359375, -83.20117187500001 34.330078125, -83.20234375000001 34.34140625, -83.20390625 34.35234375, -83.205859375 34.362890625, -83.20820312500001 34.373046875, -83.21093750000001 34.3828125, -83.21406250000001 34.3921875, -83.217578125 34.401171875, -83.221484375 34.409765625, -83.22578125000001 34.41796875, -83.23046875000001 34.42578125, -83.23554687500001 34.433203125, -83.241015625 34.440234375, -83.246875 34.446875, -83.25312500000001 34.453125, -83.25976562500001 34.458984375, -83.26679687500001 34.464453125, -83.27421875 34.46953125, -83.28203125 34.47421875, -83.29023437500001 34.478515625, -83.298828125 34.482421875, -83.3078125 34.4859375, -83.3171875 34.4890625, -83.32695312499999 34.491796875, -83.337109375 34.494140625, -83.34765625 34.49609375, -83.35859375 34.49765625, -83.369921875 34.498828125, -83.38164062499999 34.499609375, -83.39375 34.5, -83.40625 34.5, -83.418359375 34.499609375, -83.43007812500001 34.498828125, -83.44140625 34.49765625, -83.45234375 34.49609375, -83.462890625 34.494140625, -83.473046875 34.491796875, -83.48281250000001 34.4890625, -83.4921875 34.4859375, -83.501171875 34.482421875, -83.509765625 34.478515625, -83.51796875 34.47421875, -83.52578125000001 34.46953125, -83.533203125 34.464453125, -83.540234375 34.458984375, -83.546875 34.453125, -83.553125 34.446875, -83.55898437500001 34.440234375, -83.564453125 34.433203125, -83.56953125 34.42578125, -83.57421875 34.41796875, -83.578515625 34.409765625, -83.582421875 34.401171875, -83.5859375 34.3921875, -83.5890625 34.3828125, -83.59179687499999 34.373046875, -83.594140625 34.362890625, -83.59609375 34.35234375, -83.59765625 34.34140625, -83.598828125 34.330078125, -83.599609375 34.318359375, -83.6 34.306250000000006, -83.6 34.29375, -83.59960937499999 34.281640625, -83.59882812500001 34.26992187500001, -83.59765624999999 34.258593749999996, -83.59609375 34.247656250000006, -83.594140625 34.237109375, -83.59179687499999 34.226953125, -83.58906250000001 34.21718750000001, -83.58593749999999 34.207812499999996, -83.582421875 34.198828125000006, -83.578515625 34.190234375, -83.57421874999999 34.18203125, -83.56953125 34.17421875, -83.56445312499999 34.166796875, -83.558984375 34.159765625000006, -83.553125 34.153125, -83.54687499999999 34.146875, -83.54023437500001 34.141015625, -83.533203125 34.135546875, -83.52578125 34.130468750000006, -83.51796875 34.12578125, -83.50976562499999 34.121484375, -83.501171875 34.117578125, -83.4921875 34.1140625, -83.4828125 34.110937500000006, -83.473046875 34.108203125, -83.462890625 34.105859375, -83.45234375 34.10390625, -83.44140625 34.10234375, -83.430078125 34.101171875000006, -83.418359375 34.100390625, -83.40625 34.1))"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## Geometry Transformations

### geofext:constrainedDelaunay Function

In [52]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?delau
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:constrainedDelaunay(?literal) AS ?delau)
}
"""),["literal","delau"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
delau,"GEOMETRYCOLLECTION (POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.6 34.1)), POLYGON ((-83.2 34.5, -83.2 34.1, -83.6 34.1, -83.2 34.5)))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:delaunayTriangles Function

In [53]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?delau
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:delaunayTriangles(?literal) AS ?delau)
}
"""),["literal","delau"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
delau,"GEOMETRYCOLLECTION (POLYGON ((-83.6 34.5, -83.6 34.1, -83.2 34.1, -83.6 34.5)), POLYGON ((-83.6 34.5, -83.2 34.1, -83.2 34.5, -83.6 34.5)))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:scale Function

In [54]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?scale
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:scale(?literal,"2.0"^^xsd:double,"2.0"^^xsd:double,"2.0"^^xsd:double) AS ?scale)
}
"""),["literal","scale"],None)

POLYGON ((-83.79999999999998 33.900000000000006, -83 33.900000000000006, -83 34.7, -83.79999999999998 34.7, -83.79999999999998 33.900000000000006))


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
scale,"POLYGON ((-83.79999999999998 33.900000000000006, -83 33.900000000000006, -83 34.7, -83.79999999999998 34.7, -83.79999999999998 33.900000000000006))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:translate Function

In [55]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?translate
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:translate(?literal,"1.0"^^xsd:double,"1.0"^^xsd:double,"1.0"^^xsd:double) AS ?translate)
}
"""),["literal","translate"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
translate,"POLYGON ((-82.6 35.1, -82.2 35.1, -82.2 35.5, -82.6 35.5, -82.6 35.1))"


Map(center=[35.300000000000004, -82.40000000000002], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:transformCRS84 Function

In [56]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?transformed
WHERE {
  my:LExactGeom geo:asWKT ?literal .
  BIND(geof:transformCRS84(?literal) AS ?transformed)
}
"""),["literal","transformed"],None)

Variable,Value
literal,Point(-88.38 31.95)
transformed,POINT (-88.38 31.95)


Map(center=[31.95, -88.38], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:makeValid Function

In [57]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?invalidLiteral ?valid
WHERE {
  BIND("LINESTRING(0 0, 0 0)"^^geo:wktLiteral AS ?invalidLiteral)
  BIND(geof:makeValid(?invalidLiteral) AS ?valid)
}
"""),["invalidLiteral","valid"],None)

Variable,Value
invalidLiteral,"LINESTRING(0 0, 0 0)"
valid,POINT (0 0)


Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## Non-topological query functions

### geofext:compactnessRatio Function

Calculates the compactness ratio of a polygon, an indicator of polygon shape complexity.

In [58]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cratio
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:compactnessRatio(?aLiteral) as ?cratio)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cratio,0.886226925452758


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:fullyWithinDistance Function

In [59]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?fwdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:fullyWithinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?fwdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
wdistance,True


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:interpolatePoint Function

In [60]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?eLiteral ?interpolated
WHERE {
  my:E geo:hasDefaultGeometry ?eGeom .
  ?eGeom geo:asWKT ?eLiteral .
  BIND (geof:interpolatePoint(?eLiteral, "2"^^xsd:double) as ?interpolated)
}
"""),["eLiteral","interpolated"],None)

LINESTRING (-83.4 34, -83.3 34.3)
2.0


Variable,Value
eLiteral,"LineString(-83.4 34.0, -83.3 34.3)"
interpolated,POINT (-83.3 34.3)


Map(center=[34.3, -83.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [61]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:maxM(?literal) AS ?maxM)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
literal,"LineString M (-83.4 34.0 5, -83.3 34.3 10)"
maxM,10.0


Map(center=[-83.2, 34.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:metricWithinDistance Function

In [62]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral (xsd:boolean(?wdistance) as ?wddistance)
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:metricWithinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?wdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
wddistance,false


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:minimumBoundingRadius Function

In [63]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bradius
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumBoundingRadius(?aLiteral) as ?bradius)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bradius,0.28284271247460796


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:offsetCurve Function

In [65]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?offsetCurve
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:offsetCurve(?literal,1) AS ?offsetCurve)
}
"""),["literal","offsetCurve"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
offsetCurve,"LINESTRING (-84.6 34.1, -84.6 34.5, -84.58078528040322 34.69509032201613, -84.52387953251129 34.88268343236509, -84.43146961230254 35.0555702330196, -84.30710678118655 35.207106781186546, -84.1555702330196 35.33146961230255, -83.98268343236508 35.423879532511286, -83.79509032201612 35.48078528040323, -83.6 35.5, -83.2 35.5, -83.00490967798387 35.48078528040323, -82.81731656763492 35.423879532511286, -82.6444297669804 35.33146961230255, -82.49289321881345 35.207106781186546, -82.36853038769746 35.0555702330196, -82.27612046748871 34.88268343236509, -82.21921471959678 34.69509032201613, -82.2 34.5, -82.2 34.1, -82.21921471959678 33.90490967798387, -82.27612046748871 33.71731656763491, -82.36853038769746 33.5444297669804, -82.49289321881345 33.392893218813455, -82.6444297669804 33.26853038769745, -82.81731656763492 33.176120467488715, -83.00490967798387 33.11921471959677, -83.2 33.1, -83.6 33.1, -83.79509032201612 33.11921471959677, -83.98268343236508 33.176120467488715, -84.1555702330196 33.26853038769745, -84.30710678118655 33.392893218813455, -84.43146961230254 33.5444297669804, -84.52387953251129 33.71731656763491, -84.58078528040322 33.90490967798387, -84.6 34.1)"


Map(center=[34.300000000000004, -83.40000000000002], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:pointInsideCircle Function

In [66]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointINsideCircle
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  my:APointGeom geo:asWKT ?pliteral .
  BIND(geof:pointInsideCircle(?literal,?pliteral,10.0) AS ?pointInsideCircle)
}
"""),["literal"],None)

Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:pointOnSurface Function

In [67]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointOnSurface
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:pointOnSurface(?literal) AS ?pointOnSurface)
}
"""),["literal","pointOnSurface"],None)

POINT (-83.4 34.3)


Variable,Value
literal,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
pointOnSurface,POINT (-83.4 34.3)


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:sharedPaths Function

In [68]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?literal2 ?spaths
WHERE {
  my:EExactGeom geo:asWKT ?literal .
  my:EExactGeom geo:asWKT ?literal2 .
  BIND(geof:sharedPaths(?literal,?literal2) AS ?spaths)
}
"""),["literal","literal2","spaths"],None)

Variable,Value
literal,"LineString(-83.4 34.0, -83.3 34.3)"
literal2,"LineString(-83.4 34.0, -83.3 34.3)"
spaths,"GEOMETRYCOLLECTION (MULTILINESTRING ((-83.4 34, -83.3 34.3)), MULTILINESTRING EMPTY)"


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:withinDistance Function

In [69]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?wdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:withinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?wdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…